In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import joblib
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)

print(f"Using device: {DEVICE}")

IMG_H = 30
IMG_W = 640
I_MIN = 1e2
I_MAX = 1e7
N_MAX_BLOBS  = 16
LOG_I_MIN = float(np.log(I_MIN))
LOG_I_MAX = float(np.log(I_MAX))
LOG_I_RANGE = LOG_I_MAX - LOG_I_MIN  

W_EXISTS = 2.0
W_X = 5.0
W_Y = 1.0
W_I = 10.0



class BlobNet(nn.Module):
    def __init__(self, n_max: int = N_MAX_BLOBS) -> None:
        super().__init__()

        self.n_max = n_max

       
        self.encoder = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1, bias=False),
            nn.GroupNorm(4, 32),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(2, 1)),

            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.GroupNorm(4, 64),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(3, 1)),

            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.GroupNorm(4, 128),        
            nn.GELU(),
            nn.MaxPool2d(kernel_size=(5, 2)),

        )


        self.body = nn.Sequential(
            nn.Conv1d(128, 128, 5, padding=2, bias=False),
            nn.GroupNorm(8, 128),
            nn.GELU(),

            nn.Conv1d(128, 128, 5, padding=2, bias=False),
            nn.GroupNorm(8, 128),
            nn.GELU(),

            nn.Conv1d(128, 128, 3, padding=1, bias=False),
            nn.GroupNorm(8, 128),
            nn.GELU(),
        )

        
        self.loc_refine = nn.Sequential(
            nn.Conv1d(128, 64, 3, padding=1),
            nn.GroupNorm(4, 64),
            nn.GELU(),
            nn.Conv1d(64,  32, 3, padding=1),
        )
        self.pool_pos = nn.AdaptiveAvgPool1d(n_max)   

        self.head_pos = nn.Conv1d(32, 3, 1)          

      
        self.pool_int = nn.AdaptiveMaxPool1d(n_max)
        
        self.head_int = nn.Conv1d(128, 1, 1)          

    def forward(self, x: torch.Tensor):
        x = torch.asinh(x / 100.0)

        feat = self.encoder(x).squeeze(2)  
        feat = self.body(feat)             


        pos_feat = self.loc_refine(feat)   
        pos_feat = self.pool_pos(pos_feat) 
        out_pos = self.head_pos(pos_feat)  

        int_feat = self.pool_int(feat)     
        out_int = self.head_int(int_feat)

        x_pred = torch.sigmoid(out_pos[:, 0])
        y_pred = torch.sigmoid(out_pos[:, 1])
        exists_logit = out_pos[:, 2]
        int_norm_pred = torch.sigmoid(out_int[:, 0])

        return x_pred, y_pred, int_norm_pred, exists_logit






def loss_fn(outputs: tuple,targets: torch.Tensor,return_components: bool = False):

    x_pred, y_pred, int_norm_pred, exists_logit = outputs

    x_targ = targets[:, :, 0]
    y_targ = targets[:, :, 1]
    exists = targets[:, :, 3]
    denom = exists.sum() + 1e-6

  
    log_i_targ = torch.log(targets[:, :, 2].clamp(I_MIN, I_MAX))
    int_norm_targ = (log_i_targ - LOG_I_MIN) / LOG_I_RANGE

    loss_exists = F.binary_cross_entropy_with_logits(exists_logit, exists)
    loss_x = ((x_pred - x_targ) ** 2 * exists).sum() / denom
    loss_y = ((y_pred - y_targ) ** 2 * exists).sum() / denom
    loss_i = ((int_norm_pred - int_norm_targ) ** 2 * exists).sum() / denom

    total = W_EXISTS * loss_exists + W_X * loss_x + W_Y * loss_y + W_I * loss_i

    if return_components:
        return total, {
            "exists": loss_exists.item(),
            "x": loss_x.item(),
            "y": loss_y.item(),
            "i": loss_i.item(),
        }
    return total


class BlobDataset(Dataset):
    def __init__(self, data_path: str) -> None:
        raw = joblib.load(data_path)

        self.blobs = raw['blobs'].astype(np.float32)
        self.centers = raw['center']
        self.intensities = raw['intensity']
        self.classify = raw['classify']

    def __len__(self):
        return len(self.blobs)

    def __getitem__(self, idx: int):
       
        scale = np.float32(np.exp(np.random.uniform(np.log(180), np.log(2600))))
        baseline = np.float32(np.random.uniform(55, 85))

        img_pure = self.blobs[idx].reshape(30, 640) * scale

        grain = np.random.normal(0, np.float32(np.random.uniform(5.0, 9.0)), img_pure.shape).astype(np.float32)

        col_noise = np.repeat(np.random.normal(0, np.float32(np.random.uniform(4.0, 8.0)), (1, 640)).astype(np.float32),30, 0)

        col_drift = np.cumsum(np.random.normal(0, 0.18, 640)).astype(np.float32)

        col_drift = np.repeat(((col_drift - col_drift.mean()) * np.float32(np.random.uniform(0.8, 2.0)))[None],30, 0)

        row_noise = np.repeat(np.random.normal(0, np.float32(np.random.uniform(0.6, 1.5)), (30, 1)).astype(np.float32),640, 1)

        shot = np.random.normal(0,np.sqrt(np.maximum(img_pure + baseline, np.float32(1.0))) * np.float32(np.random.uniform(0.7, 1.0)),).astype(np.float32)

        hot = ((np.random.rand(*img_pure.shape) < np.random.uniform(8e-4, 2.5e-3)).astype(np.float32)* np.random.uniform(35, 120, img_pure.shape).astype(np.float32))

        dark = -((np.random.rand(*img_pure.shape) < np.random.uniform(5e-4, 1.5e-3)).astype(np.float32)* np.random.uniform(8, 25, img_pure.shape).astype(np.float32))

        noisy = np.clip(img_pure + baseline + grain + col_noise + col_drift + row_noise + shot + hot + dark,0, None).astype(np.float32)

        img_tensor = torch.from_numpy(np.ascontiguousarray(noisy)).reshape(1, 30, 640)

       
        target = torch.zeros(N_MAX_BLOBS, 4, dtype=torch.float32)
        centers = self.centers[idx].astype(np.float32)
        intens = self.intensities[idx].astype(np.float32) * scale
        exists = self.classify[idx].astype(np.float32)

        
        order = np.argsort(-intens)
        occupied = set()

        for i in order:
            if exists[i] < 0.5:
                continue

            x_px = float(centers[i][0])
            i_safe = float(np.clip(float(intens[i]), I_MIN, I_MAX))

            preferred = max(0, min(N_MAX_BLOBS - 1, int(x_px // 40)))

            candidates = [preferred]

            for d in range(1, N_MAX_BLOBS):
                if preferred - d >= 0:
                    candidates.append(preferred - d)
                if preferred + d < N_MAX_BLOBS:
                    candidates.append(preferred + d)

            for slot in candidates:
                if slot not in occupied:
                    occupied.add(slot)
                    target[slot] = torch.tensor(
                        [x_px / IMG_W, float(centers[i][1]) / IMG_H, i_safe, 1.0],
                        dtype=torch.float32)
                    break

        return img_tensor, target

sys.modules['__main__'].BlobDataset = BlobDataset



if __name__ == "__main__":

    train_ds = BlobDataset("New_MultiBlob_TRA_DAT.joblib")
    val_ds = BlobDataset("New_MultiBlob_VAL_DAT.joblib")

    train_loader = DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=0)

    val_loader = DataLoader(val_ds,batch_size=128,shuffle=False,num_workers=0)

    model = BlobNet().to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_val = float('inf')
    nan_count = 0

    for epoch in range(50):

        model.train()
        t_loss = 0.0
        t_comps = {"exists": 0.0, "x": 0.0, "y": 0.0, "i": 0.0}

        for imgs, targs in tqdm(train_loader, desc=f"Epoch {epoch+1:3d}"):
            imgs = imgs.to(DEVICE,  non_blocking=True)
            targs = targs.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            loss, comps = loss_fn(model(imgs), targs, return_components=True)

            if not torch.isfinite(loss):
                nan_count += 1
                if nan_count <= 5:
                    print(f"  [WARN] Non-finite loss={loss.item()} — skipping batch")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            t_loss += loss.item()
            for k in t_comps:
                t_comps[k] += comps[k]

    
        model.eval()
        v_loss  = 0.0
        v_comps = {"exists": 0.0, "x": 0.0, "y": 0.0, "i": 0.0}

        with torch.no_grad():
            for imgs, targs in val_loader:
                imgs  = imgs.to(DEVICE, non_blocking=True)
                targs = targs.to(DEVICE, non_blocking=True)

                vl, comps = loss_fn(model(imgs), targs, return_components=True)
                if torch.isfinite(vl):
                    v_loss += vl.item()
                    for k in v_comps:
                        v_comps[k] += comps[k]

        scheduler.step()

        n_t = len(train_loader)
        n_v = len(val_loader)
        avg_t, avg_v = t_loss / n_t, v_loss / n_v

        print(
            f"Epoch {epoch+1:3d} | "
            f"Train {avg_t:.5f}  "
            f"Val {avg_v:.5f}  " 
        )

        if avg_v < best_val:
            best_val = avg_v
            state = model._orig_mod.state_dict() if hasattr(model, '_orig_mod') else model.state_dict()
            torch.save(state, "best_blobnet.pt")
            print("  → saved best")

    print(f"\nBest val = {best_val:.5f}")